# Probability Basics

**Goal:** Simulate the law of large numbers, implement expectation and variance from scratch, and verify against PyTorch; then explore `torch.distributions` for Bernoulli and Categorical sampling.

## Configuration

Device, random seed, and default dtype come from `shared.config`, which reads `config.toml`.  
On Apple Silicon this resolves to `mps`; on CUDA machines it resolves to `cuda`; otherwise `cpu`.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Law of Large Numbers: Empirical Probability → Theoretical

We flip a biased coin (Bernoulli with p=0.3) and track how the empirical frequency converges to the true probability as n grows.

In [2]:
torch.manual_seed(42)

TRUE_P = 0.3
N_MAX = 100_000

# Draw N_MAX Bernoulli samples at once
flips = torch.bernoulli(torch.full((N_MAX,), TRUE_P, device=device))

# Cumulative mean at each step: empirical frequency after k draws
running_mean = torch.cumsum(flips, dim=0) / torch.arange(1, N_MAX + 1, device=device, dtype=torch.float32)

# ---- plot ----
fig, ax = plt.subplots(figsize=(9, 4))
ns = torch.arange(1, N_MAX + 1).cpu().numpy()
ax.semilogx(ns, running_mean.cpu().numpy(), lw=1, label="empirical p")
ax.axhline(TRUE_P, color="red", ls="--", label=f"true p = {TRUE_P}")
ax.set_xlabel("Number of flips (log scale)")
ax.set_ylabel("Cumulative frequency")
ax.set_title("Law of Large Numbers: Bernoulli(0.3)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"After {N_MAX:,} flips, empirical p = {running_mean[-1].item():.5f}  (true = {TRUE_P})")

After 100,000 flips, empirical p = 0.30048  (true = 0.3)


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_11954/3922230791.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Expectation and Variance from Scratch

For a continuous uniform distribution on [a, b]:
```
E[X] = (a + b) / 2
Var(X) = (b - a)^2 / 12
```
We sample a large set of uniform draws, compute statistics manually, then validate against `torch.mean` / `torch.var`.

In [3]:
torch.manual_seed(42)

A, B = 2.0, 8.0  # uniform distribution bounds
N = 500_000

samples = torch.rand(N, device=device) * (B - A) + A


def empirical_expectation(x: torch.Tensor) -> torch.Tensor:
    """E[X] = (1/n) * sum_i x_i."""
    return x.sum() / x.numel()


def empirical_variance(x: torch.Tensor) -> torch.Tensor:
    """Var(X) = E[(X - E[X])^2] = E[X^2] - E[X]^2."""
    mean = empirical_expectation(x)
    return empirical_expectation((x - mean) ** 2)


e_scratch = empirical_expectation(samples)
v_scratch = empirical_variance(samples)

print(f"From-scratch  E[X] = {e_scratch.item():.5f}  (theoretical = {(A + B) / 2:.1f})")
print(f"From-scratch Var(X) = {v_scratch.item():.5f}  (theoretical = {(B - A)**2 / 12:.5f})")

From-scratch  E[X] = 5.00058  (theoretical = 5.0)
From-scratch Var(X) = 2.99401  (theoretical = 3.00000)


In [4]:
# Validate against torch builtins
torch_mean = torch.mean(samples)
# torch.var uses Bessel correction by default (unbiased=True); match with unbiased=False
torch_var = torch.var(samples, unbiased=False)

assert torch.allclose(e_scratch, torch_mean, atol=1e-4), (
    f"Expectation mismatch: {e_scratch.item()} vs {torch_mean.item()}"
)
assert torch.allclose(v_scratch, torch_var, atol=1e-4), (
    f"Variance mismatch: {v_scratch.item()} vs {torch_var.item()}"
)
print("Expectation and variance match torch.mean / torch.var ✓")

Expectation and variance match torch.mean / torch.var ✓


## Idiomatic PyTorch: `torch.distributions`

`torch.distributions.Bernoulli` and `torch.distributions.Categorical` provide clean sampling and log-probability methods used throughout PyTorch's ecosystem.

In [5]:
# --- Bernoulli ---
bern = torch.distributions.Bernoulli(probs=torch.tensor(TRUE_P, device=device))
bern_samples = bern.sample((10_000,))
print(f"Bernoulli(0.3) mean from 10k samples: {bern_samples.mean().item():.4f}")

# log_prob for a single outcome
lp_heads = bern.log_prob(torch.tensor(1.0, device=device))
print(f"log P(heads) = {lp_heads.item():.4f}  (should be log(0.3) = {torch.log(torch.tensor(0.3)).item():.4f})")

# --- Categorical ---
probs = torch.tensor([0.1, 0.5, 0.4], device=device)
cat = torch.distributions.Categorical(probs=probs)
cat_samples = cat.sample((50_000,))
empirical_probs = torch.bincount(cat_samples, minlength=3).float() / 50_000
print(f"Categorical empirical probs: {empirical_probs.cpu().tolist()}")
print(f"True probs:                  {probs.cpu().tolist()}")

Bernoulli(0.3) mean from 10k samples: 0.3044
log P(heads) = -1.2040  (should be log(0.3) = -1.2040)


Categorical empirical probs: [0.10044000297784805, 0.5019800066947937, 0.39757999777793884]
True probs:                  [0.10000000149011612, 0.5, 0.4000000059604645]


## Conditional Probability and Independence Check

Verify P(A|B) = P(A ∩ B) / P(B) empirically, and check that two independent coins satisfy P(A ∩ B) ≈ P(A)·P(B).

In [6]:
torch.manual_seed(42)
N = 200_000

# Two independent coins: p_A = 0.4, p_B = 0.6
coin_A = torch.bernoulli(torch.full((N,), 0.4, device=device))
coin_B = torch.bernoulli(torch.full((N,), 0.6, device=device))

p_A = coin_A.mean()
p_B = coin_B.mean()
p_AB = (coin_A * coin_B).mean()  # intersection: both heads

# Conditional probability P(A|B) = P(A and B) / P(B)
p_A_given_B = p_AB / p_B

print(f"P(A)         = {p_A.item():.4f}  (true 0.4)")
print(f"P(B)         = {p_B.item():.4f}  (true 0.6)")
print(f"P(A and B)   = {p_AB.item():.4f}  (true 0.24)")
print(f"P(A|B)       = {p_A_given_B.item():.4f}  (should ≈ P(A) = 0.4 for independent events)")

# Independence check: P(A and B) ≈ P(A)*P(B)
assert torch.allclose(p_AB, p_A * p_B, atol=5e-3), "Independence violated!"
print("Independence check: P(A∩B) ≈ P(A)·P(B) ✓")

P(A)         = 0.3996  (true 0.4)
P(B)         = 0.5993  (true 0.6)
P(A and B)   = 0.2390  (true 0.24)
P(A|B)       = 0.3989  (should ≈ P(A) = 0.4 for independent events)
Independence check: P(A∩B) ≈ P(A)·P(B) ✓


## Takeaways

- **Law of Large Numbers:** Empirical frequencies converge to true probabilities as n → ∞. With n=100k, error is < 0.001.
- **Expectation and Variance:** Simple sum-based formulas match `torch.mean`/`torch.var` exactly.
- **`torch.distributions`:** Provides log_prob, sample, and entropy methods — the backbone of probabilistic ML in PyTorch (VAEs, REINFORCE, NLL training).
- **Conditional probability:** P(A|B) is estimated by filtering samples where B occurred. For independent events, P(A|B) ≈ P(A).
- **Takeaway for AI engineering:** Validation set metrics are random variables; more data means lower variance in your estimates, not a fixed truth.